# 02 — CAPM and the beta-target portfolio

`BetaTargetMinVar` (in `maplab.models`; BetaMinVar in tables and figures): long-only minimum variance with a floor on market β, scored by the rolling backtest against EW and 60/40. It is risk-based: Σ and β enter, no forecast of μ does.

## §1 CAPM and the idea

CAPM prices only market β: expected excess returns are proportional to β. That makes β-implied μ a tempting replacement for one year of sample means, but not a useful one for an optimizer: when the factor is itself a portfolio of the assets, the tangency on factor-implied μ holds only the factor (§2, checked in §4.2). We therefore use β the other way, as a constraint: minimum variance with a floor on market β. §4.3 asks whether size and value exposures could be targeted too.

## §2 Problem and inputs

**CAPM μ.** $\mu_i - r_f = \beta_i \lambda$, with $\beta_i = \mathrm{cov}(x_i, x_m)/\mathrm{var}(x_m)$ on daily log excess returns $x$ and $\lambda$ the market premium.

**Why it collapses.** Let the factors be portfolios of the assets, $f = W'x$, and let $B$ be estimated on the same window as $\Sigma_x$. Then $B = \Sigma_x W (W'\Sigma_x W)^{-1}$ and

$$\Sigma_x^{-1}(\mu - r_f) = \Sigma_x^{-1} B \lambda = W (W'\Sigma_x W)^{-1} \lambda .$$

The tangency portfolio holds only the assets inside the factors. For CAPM, $W = e_{SPY}$, so the tangency is 100% SPY and $\lambda$ cancels.

**The model: β-target minimum variance.**

$$\min_w \; w'\Sigma w \quad \text{s.t.} \quad \mathbf{1}'w = 1, \; w \ge 0, \; \beta'w \ge \beta^* .$$

No μ enters. $\beta^* = 0.3$ is fixed a priori; $0.5$ is a sensitivity. If GMV's own β already clears $\beta^*$, the constraint is slack and the solution is GMV.

**Inputs.** $\Sigma$ and $\beta$ come from the same trailing 252 daily log returns before the rebalance date, taken in excess of BIL via $\log(1 + r_f)$.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
from maplab import Panel, apply_style, BetaTargetMinVar, GMV, EqualWeight, FixedWeight, backtest, summary

from helpers import nb02 as h

In [ ]:
apply_style()
prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # estimation (mu, Sigma, beta)
simple_returns = ml.to_simple_returns(prices)  # compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

## §3 The class and the rolling backtest

`BetaTargetMinVar` takes Σ from the shared `Strategy._estimate` (defined in notebook 01 §3) and β from `_beta_vector` on the same window (§2, Inputs); `predict_weights` in `src/maplab/models.py` solves the §2 problem.

**Solver.** As for GMV. If GMV already meets the floor, the constraint is slack and GMV is returned (the date is logged); if no long-only portfolio reaches $\beta^*$, the class falls back to GMV and logs the date as infeasible.

**Backtest** as in notebook 01 §3.

## §4 Inputs and snapshot

The model's new input is β. §4.1 and §4.3 use the full sample, §4.2 three year-ends, §4.4–§4.5 the 2022-12-31 rebalance.

### §4.1 Betas

Full-sample OLS β of each asset's daily log excess return on MKT = SPY − BIL, with annualized excess return and $R^2$.

In [ ]:
rf_log = np.log1p(rf_daily)
excess = log_returns.sub(rf_log, axis=0).dropna(how="any")
MKT = excess["SPY"]

capm_rows = []
for tkr in ml.UNIVERSE:
    fit = ml.inference.ols(excess[tkr], MKT.to_frame("MKT"))
    ann_excess = float(excess[tkr].mean() * ml.TRADING_DAYS)
    capm_rows.append({"ticker": tkr, "beta": fit["coef"]["MKT"],
                       "ann_excess_return": ann_excess, "r2": fit["r2"]})

capm_table = pd.DataFrame(capm_rows).set_index("ticker")
capm_shown = capm_table.rename(columns={"beta": "β", "ann_excess_return": "excess return (%)", "r2": "R²"}).rename_axis(None)
ml.reporting.show_table(capm_shown, pct=["excess return (%)"], dp=2)

Security market line: realized excess return against β across the 13 assets. Dashed: cross-sectional fit. Dotted: CAPM line through the origin and SPY. One sample, no standard errors.

In [ ]:
fig, slope, intercept, spy_excess_ann = h.sml_figure(capm_table)
plt.show()

print(f"SML slope {slope:.1%} per unit β (CAPM: SPY's excess return, {spy_excess_ann:.1%}; ratio {slope / spy_excess_ann:.0%}), "
      f"intercept {intercept:.1%}")

**Reading.** The fitted line is flat: a slope of 2.3% per unit of β against CAPM's 9.3% (24% of it), and a 2.4% intercept where CAPM predicts zero. Low-β assets earned more than CAPM implies.

Rolling 252-day β of TLT, UUP, GLD and HYG; SPY is 1 by construction.

In [ ]:
fig = h.rolling_beta_figure(excess, MKT)
plt.show()

**Reading.** TLT, UUP and GLD change sign over the sample; HYG stays positive. The class re-estimates β at every rebalance on the same window as Σ.

### §4.2 CAPM μ as an input

The §2 identity on the last trading days of 2014, 2017 and 2022. Max-Sharpe on CAPM μ and on three-factor μ (MKT, TERM = TLT, CREDIT = HYG − IEF), unconstrained and long-only, with the identity's relative error.

In [ ]:
ASOF_DATES = ["2014-12-31", "2017-12-31", "2022-12-31"]
TICKERS = ml.UNIVERSE
N = len(TICKERS)

rel_errs = []
for res in h.capm_collapse(log_returns, excess, ASOF_DATES):
    asof, designs, rel1, rel2 = res["asof"], res["designs"], res["rel1"], res["rel2"]
    print(f"=== {asof.date()} ===")
    for name, w in designs.items():
        w_s = pd.Series(w, index=TICKERS)
        nz = w_s[w_s.abs() > 1e-4]
        parts = "  ".join(f"{t}={v * 100:+.1f}%" for t, v in nz.items())
        print(f"  {name}: {parts}")
    rel_errs.append((rel1, rel2))
    print()

print("identity check, max relative error over the three dates: "
      f"CAPM {max(r[0] for r in rel_errs):.1e}, three-factor {max(r[1] for r in rel_errs):.1e}")

**Reading.** CAPM μ gives 100% SPY at all three dates: CAPM's own prediction, hold the market, with SPY as the market proxy. Unconstrained, three-factor μ holds only SPY, TLT, IEF and HYG, with IEF = −HYG; the identity holds to about $10^{-15}$. To hold anything else the optimizer needs views, which notebook 03 adds to a prior implied by EW weights.

### §4.3 Factor exposures as targets

Could the portfolio also target size and value? SIZE = IWM − SPY and VALUE = IWD − IWF (panel-only ETFs, never allocatable). Each asset's full-sample excess return is regressed on MKT, SIZE and VALUE.

In [ ]:
factor_log_returns = ml.load_factor_returns()
SIZE = (factor_log_returns["IWM"] - log_returns["SPY"]).dropna()
VALUE = (factor_log_returns["IWD"] - factor_log_returns["IWF"]).dropna()

factors3 = pd.DataFrame({"MKT": MKT, "SIZE": SIZE, "VALUE": VALUE}).dropna()
excess3 = excess.loc[factors3.index, TICKERS]

factor_rows = []
for tkr in TICKERS:
    fit = ml.inference.ols(excess3[tkr], factors3)
    row = {"ticker": tkr, "r2": fit["r2"]}
    for f in ["MKT", "SIZE", "VALUE"]:
        row[f"beta_{f}"] = fit["coef"][f]
        row[f"t_{f}"] = fit["tstat"][f]
    factor_rows.append(row)

factor_table = pd.DataFrame(factor_rows).set_index("ticker")
factor_table = factor_table[["beta_MKT", "t_MKT", "beta_SIZE", "t_SIZE", "beta_VALUE", "t_VALUE", "r2"]]
# SPY regressed on MKT=SPY-BIL is degenerate (R^2=1, SPY on itself) -- its
# t-stat is numerically meaningless, not a real factor loading.
factor_table.loc["SPY", "t_MKT"] = np.nan
factor_shown = factor_table.rename(columns={
    "beta_MKT": "β MKT", "t_MKT": "t MKT", "beta_SIZE": "β SIZE", "t_SIZE": "t SIZE",
    "beta_VALUE": "β VALUE", "t_VALUE": "t VALUE", "r2": "R²"}).rename_axis(None)
ml.reporting.show_table(factor_shown, dp=2)

In [ ]:
sig = factor_table.copy()
insig_size = sig.index[sig["t_SIZE"].abs() < 2].tolist()
insig_value = sig.index[sig["t_VALUE"].abs() < 2].tolist()
print(f"|t| < 2 on SIZE ({len(insig_size)}/{N}): {insig_size}")
print(f"|t| < 2 on VALUE ({len(insig_value)}/{N}): {insig_value}")

**Reading.** SIZE is insignificant (t below 2 in absolute value) on 9 of 13 assets. VALUE is significant on 11: negative on IEF and TLT, positive on DBC and HYG. Outside equities it acts as a duration proxy. Plain OLS t-statistics, no HAC.

A cautionary design: long-only minimum variance with exact exposures $B'w = (0.5, 0.1, 0.1)$ on MKT, SIZE, VALUE, using 252-day $B$, at the same three dates.

In [ ]:
for asof, w, ok, msg in h.design_d3(log_returns, excess, factors3, ASOF_DATES):
    print(f"=== {asof.date()} ===  {'OK' if ok else 'INFEASIBLE: ' + msg}")
    if ok:
        w_s = pd.Series(w, index=TICKERS)
        nz = w_s[w_s.abs() > 1e-4]
        print("  " + "  ".join(f"{t}={v * 100:+.1f}%" for t, v in nz.items()))

**Reading.** The largest position jumps from date to date: VNQ 65% (2014), HYG 29% (2017), UUP 26% (2022). Exact targets on noisy 252-day loadings corner-hop like MSR on sample μ, so the model keeps only market β, and only as a floor.

### §4.4 Snapshot at 2022-12-31

Each class's weights at 2022-12-31 in (volatility, μ) space. μ is the 252-day sample mean, so the vertical axis is illustrative. The printed line gives each portfolio's ex-ante β.

In [ ]:
SNAP = pd.Timestamp("2022-12-31")
mu_2022, Sigma_2022 = GMV()._estimate(panel, SNAP)

snap = {
    "BetaMinVar(0.3)": BetaTargetMinVar(beta_target=0.3),
    "BetaMinVar(0.5)": BetaTargetMinVar(beta_target=0.5),
    "GMV": GMV(),
    "EqualWeight": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}
w_2022 = {name: s(panel, SNAP).reindex(Sigma_2022.columns) for name, s in snap.items()}

bmv_styles = {
    "BetaMinVar(0.3)": {"marker": "s", "s": 110, "color": ml.plotting.RUN_COLORS["BMV"], "label": "BetaMinVar(0.3)"},
    "BetaMinVar(0.5)": {"marker": "h", "s": 110, "color": ml.plotting.RUN_COLORS["BMV"], "label": "BetaMinVar(0.5)"},
    "60/40": {"marker": "D", "s": 130, "color": ml.plotting.RUN_COLORS["60/40"], "label": "60/40"},
}

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022, {name: w.to_numpy() for name, w in w_2022.items()},
    title=f"Snapshot {SNAP.date()}: BetaMinVar(0.3/0.5), GMV, EW, 60/40",
    figsize=(10, 6), legend_outside=True, styles=bmv_styles,
)
plt.show()

beta_2022 = snap["BetaMinVar(0.3)"]._beta_vector(panel, SNAP).reindex(Sigma_2022.columns).to_numpy()
print("ex-ante β, 2022-12-31: " + ", ".join(
    f"{name.replace('EqualWeight', 'EW')} {float(beta_2022 @ w.to_numpy()):.2f}" for name, w in w_2022.items()))

**Reading.** Both floors bind (β 0.30 and 0.50), against 0.04 for GMV, 0.37 for EW and 0.63 for 60/40. BetaMinVar is GMV pushed just far enough toward market risk to reach the floor.

### §4.5 Capital vs risk

Capital weights (dark) and shares of portfolio variance (light) at 2022-12-31; a negative share is a hedge. Table columns as in notebook 01 §4.5.

In [ ]:
weights_c5 = {"BetaMinVar(0.3)": w_2022["BetaMinVar(0.3)"], "GMV": w_2022["GMV"],
              "EW": w_2022["EqualWeight"], "60/40": w_2022["60/40"]}
div = ml.diagnostics.diversification_table(weights_c5, Sigma_2022).rename(columns={
    "w_UUP": "UUP weight (%)", "effN": "effective N", "n_uncorr": "uncorrelated bets",
    "max_RC_share": "largest risk share (%)", "DR": "diversification ratio", "exante_vol": "ex-ante vol (%)"})
display(ml.reporting.show_table(div, pct=["UUP weight (%)", "largest risk share (%)", "ex-ante vol (%)"], dp=2))

fig = ml.plotting.capital_vs_risk(weights_c5, Sigma_2022, title="Capital weight vs risk share, 2022-12-31 (sample Σ)")
plt.show()

**Reading.** BetaMinVar(0.3) holds 42% UUP against 58% for GMV and spreads over 2.42 uncorrelated bets (GMV 7.57, EW 1.40, 60/40 1.76). Its risk sits in equities, SPY alone 67% of variance, while UUP turns into a hedge: a negative variance share.

## §5 Out-of-sample results

Walk-forward as in notebook 01 §5. Two extra rows, β-match(β*) = β*·SPY + (1 − β*)·BIL, carry the same market β with no optimizer: a frictionless reference, rebalanced daily with no costs. The last cell counts the optimizer's slack, infeasible and retried rebalances; `tests/test_nb01_03_checks.py` rebuilds these runs and checks their Sharpe ratios against `registrations/reproduction.toml`.

In [ ]:
strategies = {
    "BetaMinVar(0.3)": BetaTargetMinVar(beta_target=0.3),
    "BetaMinVar(0.5)": BetaTargetMinVar(beta_target=0.5),
    "GMV": GMV(),
    "EqualWeight": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

scoring_start = results["GMV"]["diag"]["scoring_start"]
for beta_target in (0.3, 0.5):
    bench_ret = (beta_target * simple_returns["SPY"] + (1 - beta_target) * rf_daily).loc[scoring_start:]
    results[f"β-match({beta_target})"] = {"net": bench_ret, "wlog": None, "diag": None, "strat": None}

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"] if diag is not None else None
    win_defs = {
        "full": (net.index >= net.index.min(), None if turnover is None else turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, None if turnover is None else turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, None if turnover is None else turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel] if (turnover is not None and t_sel is not None) else None
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

full = summary_table.xs("full", level="window")
train = summary_table.xs("train (<= split)", level="window")
test = summary_table.xs("test (> split)", level="window")
overview = pd.DataFrame({
    "Sharpe full": full["sharpe"], "Sharpe train": train["sharpe"], "Sharpe test": test["sharpe"],
    "return (%)": full["ann_return"], "vol (%)": full["ann_vol"],
    "max DD (%)": full["max_dd"], "turnover (%/yr)": full["ann_turnover"],
}).rename(index={"EqualWeight": "EW"}).rename_axis(None)
shown = ml.reporting.show_table(overview, pct=["return (%)", "vol (%)", "max DD (%)", "turnover (%/yr)"], pct_dp=1, dp=2)
shown["turnover (%/yr)"] = shown["turnover (%/yr)"].astype(object).map(lambda v: "–" if pd.isna(v) else f"{v:.0f}")
shown

In [ ]:
counts = {}
for name in ("BetaMinVar(0.3)", "BetaMinVar(0.5)"):
    s = strategies[name]
    counts[name] = f"{len(s.slack_dates)}/{len(s.infeasible_dates)}/{len(s.retry_dates)} of {results[name]['diag']['n_rebalances']}"
tag = (next(iter(counts.values())) + " at both targets" if len(set(counts.values())) == 1
       else "; ".join(f"{n} {c}" for n, c in counts.items()))
print(f"BetaMinVar slack/infeasible/retry {tag}")

**Reading.** Full-sample Sharpe: 60/40 0.91, both BetaMinVar runs 0.86, β-match 0.83, EW 0.74, GMV 0.62; the BetaMinVar runs lead only in the 40-month test window (1.13, 1.14). β-match's excess return is β*·(SPY − BIL), so its 0.83 is SPY's own Sharpe at both targets, and BetaMinVar's 0.03 edge is what the optimizer adds over holding the market. BetaMinVar(0.3) runs at 5.8% volatility with a −13.9% drawdown, deeper than its β-match (−10.9%).

## §6 What drove the results

### §6.1 Weights by asset group

Target weights by group at every rebalance (dotted line: split). EW and 60/40 are fixed and appear only in the table.

In [ ]:
fig = ml.plotting.group_stackplot(results, ["BetaMinVar(0.3)", "GMV"], figsize=(9, 8), split_ts=split_ts)
plt.show()

group_means = pd.DataFrame({
    name: ml.diagnostics.weights_by_group(results[name]["wlog"]).mean()
    for name in ["BetaMinVar(0.3)", "BetaMinVar(0.5)", "GMV", "EqualWeight", "60/40"]
}).rename_axis(None).rename(columns={"EqualWeight": "EW"}).rename(index=ml.plotting.GROUP_LABELS)
ml.reporting.show_table(group_means, pct=list(group_means.columns))

**Reading.** On average BetaMinVar(0.3) holds 31% equities and 28% fx, against 7% and 37% for GMV. The floor is met by moving 24 points into equities, taken mostly from fx, i.e. UUP (9), rates (7), inflation-linked (5) and real assets (2).

### §6.2 Concentration and turnover

Effective N, largest weight and trailing-12-rebalance turnover at every rebalance; EW dashed in the effective-N and turnover panels.

In [ ]:
c3_names = ["BetaMinVar(0.3)", "BetaMinVar(0.5)", "GMV", "60/40"]
colors = {name: ml.plotting.run_color(name) for name in c3_names}
fig = ml.plotting.concentration_panel(
    results, c3_names,
    rows=("effN", "max_w", "turnover"), ref="EqualWeight", ref_label="EW", split_ts=split_ts,
    figsize=(9.5, 10.5), colors=colors,
    suptitle="BetaMinVar(0.3/0.5) vs GMV and 60/40: effective N, largest weight, turnover",
)
plt.show()

med_effn = {n: float(results[n]["wlog"].apply(ml.diagnostics.effective_n, axis=1).median())
            for n in ["BetaMinVar(0.3)", "BetaMinVar(0.5)", "GMV"]}
print("median effective N: " + "  ".join(f"{n}={v:.2f}" for n, v in med_effn.items()))

**Reading.** At 0.3 the floor spreads the book (median effective N 4.87 against 3.79 for GMV); at 0.5 it concentrates it again (3.62). BetaMinVar(0.3) turns over 59% a year, against 71% for GMV and 11% for 60/40.

### §6.3 Realized β and α

Each strategy's ex-post CAPM regression: daily simple excess return on SPY − BIL, plain OLS. The ex-ante floor bound at every rebalance (§5: never slack).

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_strategy_rows = []
for name in ["BetaMinVar(0.3)", "BetaMinVar(0.5)", "GMV", "EqualWeight", "60/40"]:
    net = results[name]["net"]
    idx = net.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    fit = ml.inference.ols(net.loc[idx] - rf_daily.loc[idx], MKT_simple.loc[idx].to_frame("MKT"))
    capm_strategy_rows.append({
        "strategy": name,
        "alpha_ann": float(fit["coef"]["alpha"] * ml.TRADING_DAYS),
        "t_alpha": float(fit["tstat"]["alpha"]),
        "beta": float(fit["coef"]["MKT"]),
        "r2": fit["r2"],
        "n": fit["n"],
    })

capm_strategy_table = pd.DataFrame(capm_strategy_rows).set_index("strategy")
capm_shown = capm_strategy_table.rename(
    columns={"alpha_ann": "α (%/yr)", "t_alpha": "t(α)", "beta": "β", "r2": "R²"},
    index={"EqualWeight": "EW"}).rename_axis(None)
shown = ml.reporting.show_table(capm_shown, pct=["α (%/yr)"], pct_dp=2, dp=2, int_cols=["n"], col_dp={"β": 3})
shown

**Reading.** BetaMinVar(0.3) realizes β 0.305 and α 0.51% a year (t 0.96). The β-match has α = 0 by construction, so this α is nearly all of BetaMinVar's edge over it; notebook 08 tests every model's α formally.

Cumulative BetaMinVar − β-match (solid: β-adjusted; dashed: total). Exploratory: the β-match pays no costs and rebalances daily.

In [ ]:
c4_pairs = [("BetaMinVar(0.3)", "β-match(0.3)"), ("BetaMinVar(0.5)", "β-match(0.5)")]
c4_series = {}
for a, b in c4_pairs:
    idx = results[a]["net"].index.intersection(results[b]["net"].index)
    c4_series[(a, b)] = results[a]["net"].loc[idx] - results[b]["net"].loc[idx]
c4_diff_capm = ml.inference.diff_regression(c4_series, c4_pairs, MKT_simple)

fig = ml.plotting.cumulative_paired(
    c4_series, c4_pairs, split_ts,
    beta={f"{a} - {b}": c4_diff_capm.loc[f"{a} - {b}", "beta_diff"] for a, b in c4_pairs},
    mkt_excess=MKT_simple,
    title="Exploratory: cumulative BetaMinVar − β-matched SPY/BIL, solid = β-adjusted, dashed = total",
)
plt.show()

**Reading.** The β-adjusted gap rose until early 2020 and has moved sideways since, test window included. It ends near 9%, about the 0.51% a year of §6.3 times 17.2 years; at 0.5 the gap is smaller throughout.

## §7 TAKEAWAY — a β floor sets market risk reliably but adds little beyond the market itself

- **CAPM μ fails algebraically.** When the factors are portfolios of the universe, the tangency stays inside them: CAPM gives 100% SPY at every date (§4.2).
- **β works as a constraint.** BetaTargetMinVar is GMV plus a β floor; at 0.3 the floor bound at all 208 rebalances and realized β was 0.305.
- **Against the benchmarks:** full-sample Sharpe 0.86 against 0.91 for 60/40 and 0.74 for EW, at 5.8% volatility against 10.1% and 7.9%.
- **Against its β-matched mix it adds little:** Sharpe 0.86 against 0.83, α 0.51% a year (t 0.96), earned before 2020, with a deeper drawdown and 59% turnover a year.
- **The security market line is flat** (slope 24% of CAPM's), and exact SIZE and VALUE targets corner-hop between dates (§4.3).

Notebook 03 replaces sample μ with a prior implied by EW weights plus momentum views (Black–Litterman).